# Kaggle End-to-End Execution: Approach 1 (Contrastive VAE)

This notebook runs the **Contrastive VAE Entity Resolution Pipeline** end-to-end on **Kaggle GPU**.
It automatically detects attached Kaggle datasets in `/kaggle/input`, fine-tunes the model on GPU, executes FAISS candidate blocking, extracts string/latent features, trains CatBoost, generates `candidate_pairs.tsv` and `matching_results.tsv` in `/kaggle/working/output/`, and validates submission format.

In [ ]:
# 1. Clone (or update) repository and install dependencies
import os
import sys

REPO_DIR = "/kaggle/working/Amazon_ML_26"
if os.path.exists("/kaggle/working"):
    if not os.path.exists(REPO_DIR):
        !git clone https://github.com/hemangjain17/Amazon_ML_26.git {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull --ff-only

# Add project root to sys.path
sys.path.insert(0, f"{REPO_DIR}/6ab10eb3b23ba_student_resource/student_resource")
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(""))))

%pip install -q torch transformers catboost unidecode indic-transliteration "rapidfuzz>=3.6"

In [ ]:
import torch
import pandas as pd
import numpy as np

try:
    from approach_1_contrastiveVAE.config import path_config, model_config, reranker_config, blocking_config
    from approach_1_contrastiveVAE.src import fast_pipeline as fp
except ImportError:
    from config import path_config, model_config, reranker_config, blocking_config
    from src import fast_pipeline as fp

print("=== PATH & KAGGLE ENVIRONMENT CONFIGURATION ===")
print(f"Dataset Directory : {path_config.dataset_dir}")
print(f"Train Directory   : {path_config.train_dir}")
print(f"Test Directory    : {path_config.test_dir}")
print(f"Output Directory  : {path_config.output_dir}")
print(f"CUDA Available    : {torch.cuda.is_available()}")
for gpu in range(torch.cuda.device_count()):
    print(f"GPU {gpu}             : {torch.cuda.get_device_name(gpu)}")
print(f"CPU cores         : {os.cpu_count()}")

## Step 1: Train Contrastive VAE Model on GPU
Fine-tunes the transformer backbone + variational bottleneck + InfoNCE metric head on ground truth triplets. **Skip this step** if the trained checkpoint is attached as a Kaggle dataset; Step 2 finds it automatically.

In [ ]:
# Skip this cell when the trained checkpoint is attached as a Kaggle dataset.
try:
    from approach_1_contrastiveVAE.src.trainer import train_contrastive_vae
except ImportError:
    from src.trainer import train_contrastive_vae

print(f"Starting Contrastive VAE training for {model_config.epochs} epochs on {model_config.device}...")

checkpoint_file = train_contrastive_vae(
    epochs=model_config.epochs,
    batch_size=model_config.batch_size,
    learning_rate=model_config.learning_rate,
    save_s3=False  # Disabled on Kaggle
)

print(f"--> Saved best Contrastive VAE checkpoint to: {checkpoint_file}")

## Step 2: Load Trained VAE & Train the CatBoost Reranker on Labelled Train Candidates
Loads the checkpoint from `/kaggle/input` (no dependency on Step 1), replicates it on both T4s, blocks a sample of train S1 against the **full** train S2+S3 pool with exact GPU top-k search, labels pairs from ground truth, fits CatBoost on GPU, and tunes the decision threshold $\tau^*$ for Macro $F_{0.5}$ on held-out train queries.

In [ ]:
import json, time

# ---- Run configuration -------------------------------------------------------
CHECKPOINT_PATH = globals().get("checkpoint_file")  # None -> auto-discover under /kaggle/input
TOP_K = blocking_config.top_k_candidates            # candidates per S1 entity (30)
N_TRAIN_QUERIES = 300_000                           # train S1 sample used to fit the reranker
REUSE_RERANKER = True                               # reuse /kaggle/working reranker on re-runs
RERANKER_PATH = os.path.join(path_config.models_dir, "reranker.cbm")
RERANKER_META = os.path.join(path_config.models_dir, "reranker_meta.json")
# ------------------------------------------------------------------------------

run_start = time.time()
fp.assert_text_parity([os.path.join(path_config.test_dir, f) for f in ("test_source1.tsv", "test_source3.tsv")])

checkpoint_path = fp.find_checkpoint(CHECKPOINT_PATH)
models = fp.load_models(checkpoint_path)

from catboost import CatBoostClassifier
if REUSE_RERANKER and os.path.exists(RERANKER_PATH) and os.path.exists(RERANKER_META):
    reranker = CatBoostClassifier()
    reranker.load_model(RERANKER_PATH)
    reranker_meta = json.load(open(RERANKER_META))
    fp.log(f"Reusing reranker {RERANKER_PATH}: {reranker_meta}")
else:
    reranker, threshold, stats = fp.train_reranker(
        models, path_config.train_dir, n_train_queries=N_TRAIN_QUERIES, top_k=TOP_K,
    )
    reranker.save_model(RERANKER_PATH)
    reranker_meta = {"threshold": threshold, "top_k": TOP_K, **stats}
    json.dump(reranker_meta, open(RERANKER_META, "w"))
optimal_threshold = reranker_meta["threshold"]
print(f"--> Reranker ready, tau* = {optimal_threshold:.2f} | {reranker_meta}")

## Step 3: Test Blocking (`candidate_pairs.tsv`) & Reranking (`matching_results.tsv`)
Encodes all test entities on both GPUs (fp16, dynamic padding), runs exact country-partitioned top-k search on GPU, writes `candidate_pairs.tsv`, then computes string/PIN/latent features in bounded chunks, scores them with CatBoost and applies $\tau^*$.

In [ ]:
test_state = fp.block_test(models, path_config.test_dir, path_config.output_dir, top_k=TOP_K)
candidate_tsv_path = test_state["candidate_path"]

matching_tsv_path = fp.rerank_test(test_state, reranker, optimal_threshold, path_config.output_dir)
print(f"--> Candidate pairs saved to: {candidate_tsv_path}")
print(f"--> Final matches saved to: {matching_tsv_path}")
print(f"Steps 2-3 wall time: {(time.time() - run_start) / 60:.1f} min")

## Step 4: Submission Validation
Run official challenge validator script (`utils/validate_submission.py`).

In [ ]:
is_valid = fp.validate_outputs(
    matching_path=os.path.join(path_config.output_dir, "matching_results.tsv"),
    candidate_path=os.path.join(path_config.output_dir, "candidate_pairs.tsv"),
    test_dir=path_config.test_dir,
)

print(f"\n==========================================================")
print(f" KAGGLE SUBMISSION VALIDATION STATUS: {'PASS' if is_valid else 'FAIL'}")
print(f"==========================================================")